# RAG Prompt Pilot — Variant Comparison

Tests 5 prompt variants on Qwen-7B / L-CiteEval HotpotQA (N=200 each).

**Root question**: can a longer-reasoning prompt recover FFT features (`dominant_freq`, `hl_ratio`, `spectral_centroid`) that collapse on short RAG traces (~40 tokens)?

| Variant | Strategy |
|---------|----------|
| V0 | Baseline (current) |
| V1 | + explicit reasoning preamble before answer |
| V2 | "Think through step by step" framing |
| V3 | + explain why each cited passage supports the claim |
| V4 | + evaluate passage relevance before answering |

**Gates (checked automatically in Cell 7)**
- G1: mean trace length of best variant > 100 tokens (vs ~40 baseline)
- G2: any variant pushes `dominant_freq` OR `hl_ratio` OR `spectral_centroid` past 60% AUROC
- G3: best variant simple-average fusion AUROC ≥ baseline + 5pp

In [ ]:
# Cell 1 — Drive mount + clone + install + imports
import os, sys, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO_DIR = '/content/hallucination_detection'
BRANCH   = 'feature/nadler-paper-alignment'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch -q && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy scikit-learn')

from spectral_utils import (
    load_model, generate_full, token_entropies_from_scores, free_memory,
    extract_all_features, sw_var_peak_adaptive, FEAT_NAMES,
    load_lciteeval, lciteeval_prompt,
    load_cache, save_cache,
    zscore, boot_auc,
)
import datasets as _ds  # freeze pyarrow before any gptqmodel install
import numpy as np
import matplotlib.pyplot as plt
import pickle

print('spectral_utils imported OK')
print(f'FEAT_NAMES ({len(FEAT_NAMES)}): {FEAT_NAMES}')

In [ ]:
# Cell 2 — Config
MODEL_ID   = 'Qwen/Qwen2.5-7B-Instruct'
TASK       = 'hotpotqa'
N_SAMPLES  = 200
TEMP       = 1.0
MAX_NEW    = 350    # slightly more headroom than Phase 10 to allow reasoning
VARIANTS   = [0, 1, 2, 3, 4]

DRIVE_BASE = '/content/drive/MyDrive/hallucination_detection'
PILOT_DIR  = f'{DRIVE_BASE}/cache/prompt_pilot'
os.makedirs(PILOT_DIR, exist_ok=True)

# Gate thresholds
G1_MIN_TRACE_LEN = 100    # tokens; baseline ~40
G2_FFT_TARGET    = 0.60   # AUROC; baseline ~0.51
G3_FUSION_DELTA  = 0.05   # pp improvement over V0

FFT_FEATS  = ['dominant_freq', 'hl_ratio', 'spectral_centroid']

print(f'PILOT_DIR : {PILOT_DIR}')
print(f'Variants  : {VARIANTS}')
print(f'Gates     : G1 > {G1_MIN_TRACE_LEN} tokens | G2 FFT > {100*G2_FFT_TARGET:.0f}% | G3 fusion +{100*G3_FUSION_DELTA:.0f}pp')

In [ ]:
# Cell 3 — Load model
mdl, tok = load_model(MODEL_ID, quantize_4bit=False)
print(f'Loaded {MODEL_ID}')

In [ ]:
# Cell 4 — Inference for all variants (checkpointed every 25 samples)
#
# Each variant saved to: PILOT_DIR/v{N}_traces.pkl
# Keys: traces (list of entropy arrays), labels (list of int), n (count)
#
# Label: 1 if any gold answer string appears in generated text (substring match).
# Skips variants whose pkl already exists — safe to re-run after disconnect.

samples = load_lciteeval(task=TASK, n_samples=N_SAMPLES)

for v in VARIANTS:
    pkl_path = os.path.join(PILOT_DIR, f'v{v}_traces.pkl')
    if os.path.exists(pkl_path):
        cached = load_cache(pkl_path)
        print(f'V{v}: already done ({cached["n"]} samples), skipping')
        continue

    print(f'\n=== Variant {v} ===')
    print(f'  Prompt preview: {lciteeval_prompt(samples[0], variant=v)[:200]}...')

    traces, labels = [], []
    for i, row in enumerate(samples):
        prompt = lciteeval_prompt(row, variant=v)
        try:
            result  = generate_full(mdl, tok, prompt,
                                    max_new_tokens=MAX_NEW, temperature=TEMP)
            entropy = token_entropies_from_scores(result['scores'])
            label   = int(any(ans.lower() in result['text'].lower()
                              for ans in row['answers']))
            traces.append(entropy)
            labels.append(label)
        except Exception as e:
            print(f'  [i={i}] error: {e}')
            continue

        if (i + 1) % 25 == 0:
            save_cache({'traces': traces, 'labels': labels, 'n': i + 1}, pkl_path)
            acc = sum(labels) / len(labels)
            avg_len = np.mean([len(t) for t in traces])
            print(f'  {i+1}/{N_SAMPLES}  acc={acc:.1%}  avg_trace_len={avg_len:.0f}')

    save_cache({'traces': traces, 'labels': labels, 'n': len(traces)}, pkl_path)
    acc = sum(labels) / len(labels)
    avg_len = np.mean([len(t) for t in traces])
    print(f'V{v} done: {len(traces)} samples  acc={acc:.1%}  avg_trace_len={avg_len:.0f}')
    print(f'  saved → {pkl_path}')

print('\nAll variants complete.')

In [ ]:
# Cell 5 — Unload model
del mdl, tok
free_memory()
print('Model unloaded.')

In [ ]:
# Cell 6 — Feature extraction for all variants
#
# Saves: PILOT_DIR/v{N}_feats.pkl  →  {'feats': dict, 'labels': array, 'trace_lengths': array}

VARIANT_DATA = {}   # v → {feats, labels, trace_lengths}

for v in VARIANTS:
    feats_pkl = os.path.join(PILOT_DIR, f'v{v}_feats.pkl')
    if os.path.exists(feats_pkl):
        VARIANT_DATA[v] = load_cache(feats_pkl)
        print(f'V{v}: loaded feats from cache')
        continue

    traces_pkl = os.path.join(PILOT_DIR, f'v{v}_traces.pkl')
    if not os.path.exists(traces_pkl):
        print(f'V{v}: traces missing, skip')
        continue

    raw           = load_cache(traces_pkl)
    traces        = raw['traces']
    labels        = np.array(raw['labels'])
    trace_lengths = np.array([len(t) for t in traces])

    feats = extract_all_features(traces)
    # sw_var_peak_adaptive override (designed for short traces)
    feats['sw_var_peak'] = sw_var_peak_adaptive(traces)

    data = {'feats': feats, 'labels': labels, 'trace_lengths': trace_lengths}
    save_cache(data, feats_pkl)
    VARIANT_DATA[v] = data
    print(f'V{v}: extracted features, avg_len={trace_lengths.mean():.0f}  acc={labels.mean():.1%}')

print('\nFeature extraction complete.')

In [ ]:
# Cell 7 — Gate checks
#
# For each variant, compute:
#   - mean trace length
#   - per-feature AUROC (flipped if < 0.5)
#   - simple-average fusion AUROC (Stage 1 proxy)
#
# Then evaluate G1 / G2 / G3 and print a clear recommendation.

def single_feat_auroc(feat_vals, labels, n_boot=300):
    vals = np.array(feat_vals, dtype=float)
    lbl  = np.array(labels, dtype=int)
    mask = ~np.isnan(vals)
    if mask.sum() < 10 or np.nanstd(vals) < 1e-9 or len(set(lbl[mask])) < 2:
        return np.nan
    a, _, _ = boot_auc(lbl[mask], vals[mask], n=n_boot)
    return max(a, 1.0 - a)   # always report as > 0.5

def fusion_auroc(feats, labels, feat_names=None, n_boot=300):
    """Simple-average fusion with supervised sign orientation."""
    if feat_names is None:
        feat_names = FEAT_NAMES
    lbl = np.array(labels, dtype=int)
    zs  = []
    for f in feat_names:
        z = zscore(np.array(feats[f], dtype=float))
        a = single_feat_auroc(feats[f], lbl, n_boot=50)
        sign = -1.0 if (np.nanmean(np.array(feats[f])[lbl == 1]) >
                        np.nanmean(np.array(feats[f])[lbl == 0])) else 1.0
        # orient: higher z → higher prob correct
        if not np.isnan(a):
            raw_a, _, _ = boot_auc(lbl, z, n=50)
            sign = 1.0 if raw_a >= 0.5 else -1.0
        zs.append(z * sign)
    fused = np.nanmean(zs, axis=0)
    mask  = ~np.isnan(fused)
    if len(set(lbl[mask])) < 2:
        return np.nan
    a, _, _ = boot_auc(lbl[mask], fused[mask], n=n_boot)
    return a

# ── Per-variant stats ─────────────────────────────────────────────────────────
RESULTS = {}
for v in VARIANTS:
    if v not in VARIANT_DATA:
        continue
    d      = VARIANT_DATA[v]
    feats  = d['feats']
    labels = d['labels']
    tlens  = d['trace_lengths']

    feat_aucs = {f: single_feat_auroc(feats[f], labels) for f in FEAT_NAMES}
    fus_auc   = fusion_auroc(feats, labels)

    RESULTS[v] = {
        'mean_len':   float(tlens.mean()),
        'p90_len':    float(np.percentile(tlens, 90)),
        'acc':        float(labels.mean()),
        'feat_aucs':  feat_aucs,
        'fusion_auc': fus_auc,
    }

# ── Table: feature AUROCs per variant ────────────────────────────────────────
WATCH_FEATS = FFT_FEATS + ['cusum_max', 'epr', 'sw_var_peak', 'trace_length']

print('=== Feature AUROC per variant (%) ===')
print(f'{"feature":<22}', end='')
for v in VARIANTS:
    print(f'{"V"+str(v):>9}', end='')
print()
print('─' * (22 + 9 * len(VARIANTS)))

for feat in WATCH_FEATS:
    marker = ' ★' if feat in FFT_FEATS else '  '
    print(f'{feat+marker:<22}', end='')
    for v in VARIANTS:
        auc = RESULTS.get(v, {}).get('feat_aucs', {}).get(feat, np.nan)
        hi  = auc > G2_FFT_TARGET if feat in FFT_FEATS else False
        tag = ' ✓' if hi else '  '
        print(f'{100*auc:>7.1f}{tag}', end='')
    print()

print()
print(f'{"mean_trace_len":<22}', end='')
for v in VARIANTS:
    print(f'{RESULTS.get(v,{}).get("mean_len", float("nan")):>9.0f}', end='')
print()

print(f'{"fusion_auc":<22}', end='')
for v in VARIANTS:
    print(f'{100*RESULTS.get(v,{}).get("fusion_auc", float("nan")):>9.1f}', end='')
print()

# ── Gate evaluation ───────────────────────────────────────────────────────────
best_len_v   = max(RESULTS, key=lambda v: RESULTS[v]['mean_len'])
best_len     = RESULTS[best_len_v]['mean_len']
baseline_len = RESULTS.get(0, {}).get('mean_len', 0)

fft_best  = {}
for feat in FFT_FEATS:
    best_v = max(RESULTS, key=lambda v: RESULTS[v]['feat_aucs'].get(feat, 0))
    fft_best[feat] = (best_v, RESULTS[best_v]['feat_aucs'].get(feat, 0))

baseline_fus  = RESULTS.get(0, {}).get('fusion_auc', 0)
best_fus_v    = max(RESULTS, key=lambda v: RESULTS[v]['fusion_auc'])
best_fus      = RESULTS[best_fus_v]['fusion_auc']

G1 = best_len > G1_MIN_TRACE_LEN
G2 = any(auc > G2_FFT_TARGET for _, auc in fft_best.values())
G3 = (best_fus - baseline_fus) >= G3_FUSION_DELTA

passed = sum([G1, G2, G3])

print()
print('=' * 52)
print('  PROMPT PILOT — GATE REPORT')
print('=' * 52)
print(f'  G1  Trace length    {"PASS ✓" if G1 else "FAIL ✗"}'
      f'  best V{best_len_v}={best_len:.0f} tok  (baseline={baseline_len:.0f}, target>{G1_MIN_TRACE_LEN})')
for feat in FFT_FEATS:
    bv, bauc = fft_best[feat]
    status = "PASS ✓" if bauc > G2_FFT_TARGET else "FAIL ✗"
    print(f'  G2  {feat:<22}{status}  best V{bv}={100*bauc:.1f}%  (target>{100*G2_FFT_TARGET:.0f}%)')
print(f'  G3  Fusion +{100*G3_FUSION_DELTA:.0f}pp         {"PASS ✓" if G3 else "FAIL ✗"}'
      f'  best V{best_fus_v}={100*best_fus:.1f}%  baseline={100*baseline_fus:.1f}%')
print('=' * 52)

if passed == 0:
    rec = (
        'ALL GATES FAILED. Short traces are a hard constraint for this model/dataset.\n'
        '  Options: (A) try a model that naturally generates longer answers,\n'
        '           (B) use chain-of-thought on a different RAG dataset,\n'
        '           (C) accept RAG as out-of-scope for spectral features.'
    )
elif G1 and not G2:
    rec = (
        f'Traces got longer (G1 PASS: V{best_len_v}={best_len:.0f} tok) but FFT features did not recover.\n'
        '  The signal gap is not purely trace-length — it may be structural (factual lookup\n'
        '  vs open-ended reasoning). Consider RAG as out-of-scope or test a harder RAG dataset.'
    )
elif G2 and not G1:
    rec = (
        'FFT features recovered (G2 PASS) without a big trace-length increase.\n'
        '  Inspect which variant won — the prompt wording itself may be shifting\n'
        '  entropy patterns regardless of length.'
    )
elif G1 and G2 and not G3:
    winning_v = max((v for v in RESULTS if any(
        RESULTS[v]['feat_aucs'].get(f, 0) > G2_FFT_TARGET for f in FFT_FEATS)), default=best_len_v)
    rec = (
        f'G1+G2 PASS: traces longer and FFT features recovered on V{winning_v}.\n'
        f'  Fusion did not improve by +5pp yet (G3 FAIL): {100*best_fus:.1f}% vs baseline {100*baseline_fus:.1f}%.\n'
        f'  Recommended: adopt V{winning_v} as the new RAG prompt and re-run Phase 10 RAG\n'
        '  cells with N=200 for a full bootstrapped AUROC comparison.'
    )
else:
    winning_v = best_fus_v
    rec = (
        f'ALL GATES PASSED. V{winning_v} is the winning prompt variant.\n'
        f'  Fusion AUROC: {100*best_fus:.1f}% (baseline {100*baseline_fus:.1f}%, delta +{100*(best_fus-baseline_fus):.1f}pp).\n'
        f'  Recommended: replace lciteeval_prompt(row) with lciteeval_prompt(row, variant={winning_v})\n'
        '  in all Phase 10 RAG inference cells and re-run for final numbers.'
    )

print()
print('RECOMMENDATION:')
for line in rec.split('\n'):
    print(' ', line)
print()

save_cache(RESULTS, os.path.join(PILOT_DIR, 'pilot_results.pkl'))
print(f'Saved pilot_results.pkl')

In [ ]:
# Cell 8 — Plots

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: FFT feature AUROCs per variant
for feat, color in zip(FFT_FEATS, ['#1f77b4', '#ff7f0e', '#2ca02c']):
    ys = [100 * RESULTS.get(v, {}).get('feat_aucs', {}).get(feat, np.nan) for v in VARIANTS]
    axes[0].plot(VARIANTS, ys, 'o-', color=color, lw=2, ms=8, label=feat)
axes[0].axhline(60, color='#d7301f', lw=1.2, ls='--', label='G2 target (60%)')
axes[0].set_xlabel('Prompt variant')
axes[0].set_ylabel('AUROC (%)')
axes[0].set_title('FFT feature recovery (★ = gate target)', fontsize=11)
axes[0].set_xticks(VARIANTS)
axes[0].legend(fontsize=9)
axes[0].set_ylim(40, 80)

# Middle: mean trace length per variant
lens = [RESULTS.get(v, {}).get('mean_len', np.nan) for v in VARIANTS]
axes[1].bar(VARIANTS, lens, color='#4292c6', width=0.6)
axes[1].axhline(G1_MIN_TRACE_LEN, color='#d7301f', lw=1.2, ls='--',
                label=f'G1 target ({G1_MIN_TRACE_LEN} tok)')
for x, y in zip(VARIANTS, lens):
    axes[1].text(x, y + 2, f'{y:.0f}', ha='center', fontsize=9)
axes[1].set_xlabel('Prompt variant')
axes[1].set_ylabel('Mean trace length (tokens)')
axes[1].set_title('Trace length per variant', fontsize=11)
axes[1].set_xticks(VARIANTS)
axes[1].legend(fontsize=9)

# Right: fusion AUROC per variant
fus = [100 * RESULTS.get(v, {}).get('fusion_auc', np.nan) for v in VARIANTS]
base_fus = 100 * RESULTS.get(0, {}).get('fusion_auc', 0)
axes[2].bar(VARIANTS, fus, color='#41ab5d', width=0.6)
axes[2].axhline(base_fus, color='#888', lw=1.2, ls='--', label=f'V0 baseline ({base_fus:.1f}%)')
axes[2].axhline(base_fus + 5, color='#d7301f', lw=1.2, ls='--',
                label=f'G3 target (+5pp = {base_fus+5:.1f}%)')
for x, y in zip(VARIANTS, fus):
    axes[2].text(x, y + 0.3, f'{y:.1f}', ha='center', fontsize=9)
axes[2].set_xlabel('Prompt variant')
axes[2].set_ylabel('Fusion AUROC (%)')
axes[2].set_title('Simple-average fusion AUROC', fontsize=11)
axes[2].set_xticks(VARIANTS)
axes[2].legend(fontsize=9)

fig.suptitle('Prompt Pilot — RAG Variant Comparison (Qwen-7B / HotpotQA)', fontsize=13, fontweight='bold')
fig.tight_layout()
plot_path = os.path.join(PILOT_DIR, 'pilot_summary.png')
fig.savefig(plot_path, bbox_inches='tight', dpi=120)
plt.show()
print(f'Saved {plot_path}')